In [1]:
"""
==================================================================================
 CUSTOMER RETENTION INTELLIGENCE PLATFORM
 Phase 2: Data Preparation / Cleaning -> Primary Dataset (E_Comm-Table_1.csv)
==================================================================================
 Purpose : Clean the raw e-commerce churn dataset so it is ready for feature
           engineering and modeling. This script performs:
               1. Standardization of inconsistent category labels
               2. Missing value treatment (median imputation for numeric columns)
               3. Duplicate record removal
               4. Data type corrections
               5. Final validation + export of the cleaned dataset

 Note: This is Phase 2 (Data Preparation), following the Phase 1 Data
       Understanding script. No rows are dropped due to missing values, since
       imputation preserves sample size for a dataset of this size (5,630 rows).
==================================================================================
"""

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


def section(title: str) -> None:
    print("\n" + "=" * 90)
    print(f" {title}")
    print("=" * 90)


# ----------------------------------------------------------------------------------
# 1. LOAD RAW DATA
# ----------------------------------------------------------------------------------
section("1. LOAD RAW DATA")

RAW_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/RAW/primary Data/E Comm-Table 1.csv"
CLEANED_PATH = "/Users/meetnakrani/Library/Mobile Documents/com~apple~CloudDocs/Customer-Retention-intelligence-Platform/DataSet/cleaned/E_Comm_Cleaned.csv"

df = pd.read_csv(RAW_PATH)
print(f"Raw dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")


# ----------------------------------------------------------------------------------
# 2. STANDARDIZE INCONSISTENT CATEGORY LABELS
# ----------------------------------------------------------------------------------
section("2. STANDARDIZE CATEGORY LABELS")

print("Before standardization:")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"  {col}: {sorted(df[col].unique())}")

# Merge duplicate/near-duplicate category spellings into a single canonical label
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Mobile Phone": "Phone"
})

df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "CC": "Credit Card",
    "COD": "Cash on Delivery"
})

df["PreferedOrderCat"] = df["PreferedOrderCat"].replace({
    "Mobile": "Mobile Phone"
})

print("\nAfter standardization:")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"  {col}: {sorted(df[col].unique())}")


# ----------------------------------------------------------------------------------
# 3. DUPLICATE RECORD REMOVAL
# ----------------------------------------------------------------------------------
section("3. DUPLICATE RECORD REMOVAL")

full_dupes = df.duplicated().sum()
id_dupes = df["CustomerID"].duplicated().sum()
print(f"Fully duplicated rows before cleaning : {full_dupes}")
print(f"Duplicated CustomerID before cleaning  : {id_dupes}")

before_rows = df.shape[0]
df = df.drop_duplicates()                                  # safety net, even if 0 found
df = df.drop_duplicates(subset="CustomerID", keep="first")  # guard against repeat IDs
after_rows = df.shape[0]

print(f"Rows removed as duplicates             : {before_rows - after_rows}")
print(f"Rows remaining                         : {after_rows}")


# ----------------------------------------------------------------------------------
# 4. MISSING VALUE TREATMENT
# ----------------------------------------------------------------------------------
section("4. MISSING VALUE TREATMENT")

missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
print("Missing values before imputation:")
print(missing_before)

# All missing columns are numerical and right-skewed (confirmed in Phase 1) ->
# median imputation is more robust to outliers than mean imputation here.
numeric_missing_cols = missing_before.index.tolist()

for col in numeric_missing_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  Filled '{col}' missing values with median = {median_val}")

missing_after = df.isnull().sum().sum()
print(f"\nTotal missing values after imputation: {missing_after}")


# ----------------------------------------------------------------------------------
# 5. DATA TYPE CORRECTIONS
# ----------------------------------------------------------------------------------
section("5. DATA TYPE CORRECTIONS")

# Columns that were float only because of missing values -> safe to convert to int now
int_convertible_cols = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount", "DaySinceLastOrder"
]
for col in int_convertible_cols:
    df[col] = df[col].astype(int)

print("Updated dtypes:")
print(df.dtypes)


# ----------------------------------------------------------------------------------
# 6. FINAL VALIDATION
# ----------------------------------------------------------------------------------
section("6. FINAL VALIDATION")

print(f"Final shape                : {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Remaining missing values   : {df.isnull().sum().sum()}")
print(f"Remaining duplicate rows   : {df.duplicated().sum()}")
print(f"Remaining duplicate IDs    : {df['CustomerID'].duplicated().sum()}")

print("\nCategory sanity check (should show only canonical labels):")
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
    print(f"  {col}: {sorted(df[col].unique())}")


# ----------------------------------------------------------------------------------
# 7. EXPORT CLEANED DATASET
# ----------------------------------------------------------------------------------
section("7. EXPORT CLEANED DATASET")

df.to_csv(CLEANED_PATH, index=False)
print(f"Cleaned dataset saved to: {CLEANED_PATH}")

section("DATA CLEANING COMPLETE")
print("Next step: Feature engineering (encoding categorical variables, scaling)")
print("and train/test split before model building.")


 1. LOAD RAW DATA
Raw dataset loaded: 5630 rows, 20 columns

 2. STANDARDIZE CATEGORY LABELS
Before standardization:
  PreferredLoginDevice: ['Computer', 'Mobile Phone', 'Phone']
  PreferredPaymentMode: ['CC', 'COD', 'Cash on Delivery', 'Credit Card', 'Debit Card', 'E wallet', 'UPI']
  PreferedOrderCat: ['Fashion', 'Grocery', 'Laptop & Accessory', 'Mobile', 'Mobile Phone', 'Others']

After standardization:
  PreferredLoginDevice: ['Computer', 'Phone']
  PreferredPaymentMode: ['Cash on Delivery', 'Credit Card', 'Debit Card', 'E wallet', 'UPI']
  PreferedOrderCat: ['Fashion', 'Grocery', 'Laptop & Accessory', 'Mobile Phone', 'Others']

 3. DUPLICATE RECORD REMOVAL
Fully duplicated rows before cleaning : 0
Duplicated CustomerID before cleaning  : 0
Rows removed as duplicates             : 0
Rows remaining                         : 5630

 4. MISSING VALUE TREATMENT
Missing values before imputation:
Tenure                         264
WarehouseToHome                251
HourSpendOnApp        